<a href="https://colab.research.google.com/github/Akpati-Lucan/algoverse-research/blob/master/Weight_Wise_Loss_SImulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experimental Weight-Wise Loss Guided Hebbian Learning

## A Backpropagation-Free Learning Rule Using Local Loss Simulation

This notebook investigates a new Hebbian learning approach.

Traditional Hebbian learning updates weights using only neuron activity:

\[
\Delta w_{ij} = \eta x_i y_j
\]

However, this does not tell a neuron whether its update improves the overall objective.

This experiment introduces a different idea:

For each weight:

1. Temporarily modify only that weight.
2. Evaluate the loss change.
3. Estimate whether increasing or decreasing the weight improves performance.
4. Update only that weight.

The objective is to test whether individual weights can discover their own contribution to the loss without using backpropagation.

# Import Libraries

In [ ]:
import torch
from torch import nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

import numpy as np
import time

# Device Setup

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print(device)

## Dataset

To make weight-wise loss simulation computationally feasible, we intentionally use a small dataset.

The full MNIST dataset contains 60,000 images.

Here we only use:

- 200 training examples
- 50 testing examples

This allows us to experiment with the learning rule before optimizing the algorithm.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor()
])

mnist_train = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

mnist_test = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_subset = Subset(
    mnist_train,
    range(200)
)

test_subset = Subset(
    mnist_test,
    range(50)
)

train_loader = DataLoader(
    train_subset,
    batch_size=1,
    shuffle=True
)

test_loader = DataLoader(
    test_subset,
    batch_size=1,
    shuffle=False
)

## One-Hot Encoding

The network outputs 10 values, one for each digit.

Example:

Digit 3 becomes:

\[
[0,0,0,1,0,0,0,0,0,0]
\]

This allows us to measure prediction error using a loss function.

In [ ]:
def one_hot(y, classes=10):
    vector = torch.zeros(classes)
    vector[y] = 1
    return vector

## Experimental Network

We intentionally avoid deep networks.

The purpose is not to achieve state-of-the-art accuracy.

The purpose is testing whether a single weight can determine its own update direction.

In [ ]:
class SmallHebbianNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.weights = torch.randn(
            784,
            10
        ) * 0.01

    def forward(self,x):
        return x @ self.weights

## Simulated Objective Function

Although the learning rule does not use gradients, we still need a scalar objective.

The loss acts as an evaluator:

"Did changing this weight make the network better or worse?"

Here we use Mean Squared Error.

In [ ]:
loss_function = nn.MSELoss()

## Proposed Learning Rule

For every weight:

1. Measure current loss.

\[
L(w)
\]


2. Perturb the weight.

\[
w'=w+\epsilon
\]


3. Measure new loss.

\[
L(w')
\]


4. Estimate direction:

\[
\Delta L=L(w')-L(w)
\]


If loss decreases:

increase the weight.

If loss increases:

decrease the weight.


This approximates:

\[
\frac{\partial L}{\partial w}
\approx
\frac{L(w+\epsilon)-L(w)}{\epsilon}
\]


without calculating gradients.

In [ ]:
def simulated_loss_update(
    model,
    x,
    target,
    lr=0.01,
    epsilon=0.001
):
    current_output = model(x)
    current_loss = loss_function(
        current_output,
        target
    )
    rows, cols = model.weights.shape

    for i in range(rows):
        for j in range(cols):
            original = model.weights[i,j].item()

            # try increasing weight
            model.weights[i,j] = original + epsilon

            new_loss = loss_function(
                model(x),
                target
            )

            if new_loss < current_loss:
                model.weights[i,j] += lr
            else:
                model.weights[i,j] -= lr

            # restore numerical stability
            model.weights[i,j] = model.weights[i,j].detach()

## Training

Unlike backpropagation:

- no optimizer,
- no gradients,
- no backward pass.

The weight itself experiments with possible changes.

In [ ]:
model = SmallHebbianNetwork()
epochs = 3
start = time.time()
for epoch in range(epochs):
    for image,label in train_loader:
        x = image.view(1,-1)
        y = one_hot(label.item())
        y = y.unsqueeze(0)
        simulated_loss_update( model, x, y)
    print(f"Epoch {epoch+1} complete")

print("Training time:", time.time()-start)

## Testing Accuracy

After training, we measure whether the loss-guided Hebbian rule learned useful features.

In [ ]:
correct = 0
total = 0
with torch.no_grad():
    for image,label in test_loader:
        x=image.view(1,-1)
        prediction = model(x)
        predicted = torch.argmax(prediction)
        if predicted.item()==label.item():
            correct+=1
        total+=1
print("Accuracy:",correct/total*100,"%")